In [24]:
%%capture
!pip -q install openai pdfplumber numpy "gradio>=4.44,<6"

In [33]:
from google.colab import userdata

API_KEY = userdata.get("NEXUS_API_KEY")   # Secrets la neenga save panna peyar

BASE_URL    = "https://nexusapi.navigatelabs.ai"
MODEL_NAME  = "nova-micro"        # Cell A output paathu apuram podalaam
EMBED_MODEL = "text-embedding-3-small"   # Cell A output paathu apuram podalaam
TOP_K = 3
print("Settings loaded ✅")

Settings loaded ✅


In [34]:
import json, time, re, io, html
import numpy as np
import pdfplumber
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)


def safe_err(e) -> str:
    """Error text with the API key scrubbed out."""
    return str(e).replace(API_KEY, "***")


def llm_json(prompt: str, retries: int = 3):
    last = None
    for attempt in range(1, retries + 1):
        try:
            messages = [{"role": "user", "content": prompt}]
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME, messages=messages,
                    response_format={"type": "json_object"},
                )
            except Exception as e:
                if "response_format" in str(e).lower():
                    resp = client.chat.completions.create(model=MODEL_NAME, messages=messages)
                else:
                    raise
            raw = resp.choices[0].message.content
            raw = raw.replace("```json", "").replace("```", "").strip()
            return json.loads(raw)
        except Exception as e:
            last = e
            if "PerDay" in str(e):
                break
            time.sleep(3 * attempt)
    raise RuntimeError(f"Model call failed: {safe_err(last)}")


def embed(texts):
    vecs = []
    for i in range(0, len(texts), 50):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 50])
        vecs.extend(d.embedding for d in r.data)
    m = np.array(vecs, dtype="float32")
    return m / np.linalg.norm(m, axis=1, keepdims=True)


print("Client ready ✅")

Client ready ✅


In [8]:
RULES_TEXT = """
Bond / service agreement: Under Section 74 of the Indian Contract Act, 1872, if a contract fixes a penalty for breach, the court allows only reasonable compensation not exceeding the stated amount. A bond amount far higher than the employer's actual loss (for example real training cost) can be challenged as a penalty. The employer should be able to show a training cost breakup.

Bond duration: Service bonds are generally accepted only when the employer spends real money on specific training or sponsorship. Very long bonds (more than 2 years) with heavy penalty (more than about 3 months of salary) are considered harsh and one-sided. Reasonable practice: 6 to 12 months with penalty proportional to the cost.

Pro-rata bond recovery: A fair bond reduces the recoverable amount for every month the employee has already served. A flat penalty regardless of months served is a red flag.

Non-compete after employment: Section 27 of the Indian Contract Act, 1872 makes agreements in restraint of trade void. A clause stopping an employee from joining a competitor or working in the same field after leaving the job is generally not enforceable in India. Restrictions during the employment period are allowed. Reasonable confidentiality and non-solicitation of trade secrets are usually acceptable.

Notice period: Notice periods in India are usually 1 to 3 months by contract. Anything above 90 days for a fresher or junior role is considered long. Notice should apply equally to employer and employee, and the employee should not be asked to pay more than the salary for the notice period (notice pay in lieu).

Holding original certificates: Employers holding original educational certificates of an employee is widely considered an unfair practice, and the employee should not be forced to submit originals as security. Verification of originals is fine, but they should be returned immediately. Ask for a written list of documents and the return date.

Unpaid training / unpaid internship: Under the Apprentices Act, 1961, apprentices must be paid a stipend as per notified rates. Long unpaid training for a regular employee role is a red flag. The training period should be clearly defined, with a stipend or salary and written confirmation of employment after completion. Minimum Wages laws apply to employees.

Probation period: Probation of 3 to 6 months is common. Salary during probation should not be unreasonably lower than the confirmed salary, and the terms for confirmation should be written clearly. Unlimited or repeatedly extendable probation is a red flag.

Salary deductions and full and final settlement: Under the Payment of Wages Act, 1936 deductions from wages are allowed only for specified reasons. Wages and dues must be paid on exit. Withholding salary, final settlement or relieving and experience letters as a penalty for leaving early is a red flag.

One-sided termination: The employer terminating at any time without notice while the employee must give long notice is one-sided. Termination for misconduct should follow a fair process. Termination clauses should state notice and reasons clearly for both sides.

Working hours and overtime: State Shops and Establishments Acts generally limit working hours (commonly 8 to 9 hours per day and 48 hours per week) and require overtime pay. Clauses like "working hours are flexible as per company need" without overtime or comp-off should be clarified.

Variable pay and CTC: CTC is not the in-hand salary. Offer letters should clearly state fixed salary, variable pay, PF and gratuity components. Clauses letting the employer reduce or withhold variable pay at its sole discretion should be clarified.

Confidentiality and intellectual property: Confidentiality of company data during and after employment is reasonable. Clauses claiming ownership over everything the employee creates, including personal projects outside work hours and without company resources, are overly broad.

Joining bonus and relocation clawback: Recovery of a joining bonus or relocation reimbursement if the employee leaves early is acceptable only if the period and amount are reasonable and reduce pro-rata over time.

Leave and benefits: Leave policy, holidays, PF, gratuity and insurance should be mentioned or referenced to a policy document. Missing details should be asked in writing before signing.
"""

KB_CHUNKS = [p.strip() for p in re.split(r"\n\s*\n", RULES_TEXT) if len(p.strip()) > 30]
KB_SOURCES = ["starter_rules"] * len(KB_CHUNKS)
print(len(KB_CHUNKS), "rules loaded")

15 rules loaded


In [16]:
KB_VECS = None


def build_kb():
    global KB_VECS
    KB_VECS = embed(KB_CHUNKS)
    return len(KB_CHUNKS)


def retrieve_many(queries, k=TOP_K):
    q = embed(queries)
    sims = q @ KB_VECS.T
    out = []
    for row in sims:
        idx = np.argsort(-row)[:k]
        out.append([{"source": KB_SOURCES[i], "text": KB_CHUNKS[i]} for i in idx])
    return out


print("Chunks indexed:", build_kb())
for r in retrieve_many(["Candidate must serve 24 months bond or pay Rs 2,00,000"])[0]:
    print("-", r["source"], "|", r["text"][:100].replace("\n", " "))

Chunks indexed: 17
- starter_rules | Bond / service agreement: Under Section 74 of the Indian Contract Act, 1872, if a contract fixes a p
- starter_rules | Bond duration: Service bonds are generally accepted only when the employer spends real money on spec
- sample_offer.pdf | OFFER OF EMPLOYMENT - Acme Tech Solutions Pvt Ltd Dear Candidate, we are pleased to offer you the po


In [17]:
from google.colab import files


def merge_chunks(text, max_chars=900):
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) + 2 <= max_chars:
            cur = (cur + "\n\n" + p).strip()
            continue
        if cur:
            chunks.append(cur)
        while len(p) > max_chars:
            chunks.append(p[:max_chars])
            p = p[max_chars - 100:]
        cur = p
    if cur:
        chunks.append(cur)
    return chunks


uploaded = files.upload()
for name, data in uploaded.items():
    if name.lower().endswith(".pdf"):
        with pdfplumber.open(io.BytesIO(data)) as pdf:
            text = "\n\n".join((pg.extract_text() or "") for pg in pdf.pages)
    else:
        text = data.decode("utf-8", errors="ignore")
    new = merge_chunks(text)
    KB_CHUNKS.extend(new)
    KB_SOURCES.extend([name] * len(new))
    print(f"{name}: {len(new)} chunks added")

print("Total chunks indexed:", build_kb())

Saving sample_offer.pdf to sample_offer (1).pdf
sample_offer (1).pdf: 2 chunks added
Total chunks indexed: 19


In [11]:
EXTRACT_PROMPT = """You are a legal document parser. Below is the text of an offer letter /
internship agreement. Split it into individual clauses.

Rules:
- Do NOT change or summarise the clause text. Copy it exactly as written.
- Ignore headers, footers, page numbers and signature blocks.
- Give each clause a category from this list:
  ["bond", "notice_period", "unpaid_training", "non_compete", "certificate_hold",
   "salary", "probation", "termination", "working_hours", "leave", "confidentiality", "other"]
- Return ONLY a JSON object in this format:
{"clauses": [{"id": 1, "category": "bond", "text": "Candidate must serve 24 months or pay Rs 2,00,000."}]}

OFFER LETTER TEXT:
<<TEXT>>
"""


def pdf_to_text(path):
    with pdfplumber.open(path) as pdf:
        text = "\n".join((pg.extract_text() or "") for pg in pdf.pages).strip()
    if not text:
        raise ValueError("No text could be extracted. This looks like a scanned PDF. Please upload a text-based PDF.")
    return text


def extract_clauses(text):
    data = llm_json(EXTRACT_PROMPT.replace("<<TEXT>>", text))
    clauses = data["clauses"] if isinstance(data, dict) else data
    for i, c in enumerate(clauses, 1):
        c["id"] = c.get("id", i)
        c.setdefault("category", "other")
    return clauses


print("Extractor ready ✅")

Extractor ready ✅


In [39]:
RISK_PROMPT = """You are ClauseCop, an assistant that helps Indian freshers understand risky clauses
in offer letters and internship agreements. You give awareness, NOT legal advice.

Below are numbered clauses, each with an automatic category tag, its text and REFERENCE RULES
retrieved from labour-law / HR guideline documents. Assess EVERY clause.

STRICT RULES:
- The category tag is only a hint and may be wrong. Judge ONLY by the clause text.
- Never invent facts. Every number, duration or amount in your reason must appear in the clause text.
- "evidence" must be an exact phrase copied from the clause text (max 15 words) that supports your rating.
- Never classify a clause as High Risk without clear evidence
  of an actual risk in the clause text.
- If training is explicitly paid at full salary, classify it as Low.
- If overtime is explicitly compensated as per law,
  do not describe it as unpaid overtime.
- Never assume unpaid training when the clause says paid training.
- Never assume unpaid overtime when compensation is mentioned.
- If the clause is fair or contains standard employment conditions,
  classify it as Low unless another specific risk is present.
- The reason must explain the actual clause, not an imagined situation.

RISK GUIDELINES:
- Bond: >12 months OR penalty > 3 months' salary OR no training-cost breakup = High.
  6-12 months with proportional penalty = Medium. No bond / reasonable = Low.
- Notice period: >90 days = High; 60-90 days = Medium; <=30 days = Low.
- Unpaid training / unpaid probation beyond 1 month = High. PAID training = Low.
- Non-compete AFTER employment ends = High. Reasonable confidentiality / non-solicit = Low.
- Holding original certificates for the bond period = High. Verifying and returning them the same day = Low.
- One-sided termination = Medium or High.
- Termination where BOTH parties give equal notice (e.g. 30 days or salary in lieu) = Low.
- Standard working hours (about 8-9 hours/day, 5 days/week) with overtime paid as per law = Low.
- Normal salary, leave, working hours, probation of 3 months with written confirmation = Low
  unless unusual (unpaid overtime, variable pay at sole discretion = Medium).
- If unsure choose Medium and explain why.

CLAUSES:
<<CLAUSES>>

Return ONLY a JSON object with exactly one entry per clause, in the same order:
{"results": [
  {"id": <clause number>,
   "risk": "High" | "Medium" | "Low",
   "evidence": "exact phrase copied from the clause text",
   "reason": "1-2 short sentences in English",
   "explanation_en": "simple 2-3 sentence explanation for a fresher",
   "explanation_ta": "the same explanation in simple spoken-style Tamil (Tamil script)",
   "ask_hr": "one specific question the candidate should ask HR"}
]}
"""
def _norm(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()


def _nums(s):
    return set(re.findall(r"\d+", str(s).replace(",", "")))


def _verified(d, clause_text, rules):
    """Hallucination guard: numbers in the explanation must come from the clause or the reference rules."""
    allowed = _nums(clause_text)
    for r in rules:
        allowed |= _nums(r["text"])
    said = _nums(d.get("reason", "")) | _nums(d.get("explanation_en", ""))
    if not said <= allowed:
        return False
    ev = _norm(d.get("evidence", ""))
    if ev and ev not in _norm(clause_text):
        words = re.findall(r"\w+", ev)
        text_words = set(re.findall(r"\w+", _norm(clause_text)))
        if not words or sum(w in text_words for w in words) / len(words) < 0.7:
            return False
    return True
from concurrent.futures import ThreadPoolExecutor

def _assess_one(c, rules):
    rtxt = "\n".join(
        f"   - [{r['source']}] {r['text']}" for r in rules
    )

    block = f"""### Clause {c['id']}
Category: {c['category']}
Exact clause text: {c['text']}
Reference rules:
{rtxt}
"""

    data = llm_json(RISK_PROMPT.replace("<<CLAUSES>>", block))
    items = data["results"] if isinstance(data, dict) else data
    d = items[0] if items else {}

    risk = str(d.get("risk", "Medium")).strip().capitalize()

    if risk not in ("High", "Medium", "Low"):
        risk = "Medium"

    reason = d.get("reason", "")
    evidence = d.get("evidence", "")
    explanation_en = d.get("explanation_en", "")
    explanation_ta = d.get("explanation_ta", "")

    # RULE 1: Clearly paid training should be LOW risk
    training_paid = re.search(
        r"\b(paid training|paid on-the-job training|"
        r"training at full salary|training with stipend)\b",
        c["text"], re.I
    )

    training_unpaid = re.search(
        r"\b(unpaid training|without pay|without salary|"
        r"no stipend|unpaid internship)\b",
        c["text"], re.I
    )

    if training_paid and not training_unpaid:
        risk = "Low"
        evidence = training_paid.group(0)
        reason = "The clause clearly confirms that training is paid."
        explanation_en = (
            "You will receive payment during the training period. "
            "This is a positive condition and does not indicate unpaid training risk."
        )
        explanation_ta = (
            "Training period-la ungalukku salary kidaikkum. "
            "Idhu oru positive condition. Unpaid training risk inga illa."
        )

    # RULE 2: Overtime compensated as per law = LOW risk
    overtime_paid = re.search(
        r"overtime.{0,100}(compensated|paid|payment).{0,50}(law|approved)?",
        c["text"], re.I
    )

    standard_hours = re.search(
        r"\b(8|9)\s*hours?\s*per\s*day\b",
        c["text"], re.I
    )

    if overtime_paid and standard_hours:
        risk = "Low"
        evidence = overtime_paid.group(0).strip()
        reason = (
            "The clause specifies standard working hours and confirms "
            "that overtime will be compensated as per law."
        )
        explanation_en = (
            "Your working hours are clearly mentioned. "
            "The company also confirms overtime compensation as per law."
        )
        explanation_ta = (
            "Ungaloda working hours clear-ah mention pannirukkanga. "
            "Extra time work panna law-padi compensation kuduppanga."
        )

    # Verify the final explanation against the actual clause
    d["evidence"] = evidence
    d["reason"] = reason
    d["explanation_en"] = explanation_en

    verified = _verified(d, c["text"], rules)

    return {
        **c,
        "risk": risk,
        "verified": verified,
        "reason": reason,
        "explanation_en": explanation_en,
        "explanation_ta": explanation_ta,
        "ask_hr": d.get("ask_hr", ""),
        "references": rules,
    }






In [40]:
import logging
import gradio as gr

# Hide harmless background server logs
for _n in ("uvicorn", "uvicorn.error", "uvicorn.access", "fastapi", "httpx"):
    logging.getLogger(_n).disabled = True

CSS = """
.gradio-container {max-width: 1180px !important; margin: auto;}
.cc-hero {background: linear-gradient(135deg,#0f172a 0%,#1e3a8a 60%,#7f1d1d 100%);
          padding: 30px 34px; border-radius: 18px; margin-bottom: 14px;}
.cc-chip {display:inline-block; background: rgba(255,255,255,.14); padding: 5px 13px; border-radius: 999px;
          font-size: 12px; margin: 0 8px 6px 0; color:#fff;}
.cc-stats {display:grid; grid-template-columns: repeat(4,1fr); gap: 12px; margin-bottom: 12px;}
.cc-stat {background:#fff; border:1px solid #e5e7eb; border-radius:14px; padding:16px; text-align:center;}
.cc-banner {padding: 13px 18px; border-radius: 12px; margin-bottom: 16px; font-weight: 600;}
.cc-card {background:#fff; border:1px solid #e5e7eb; border-left: 6px solid #9ca3af;
          border-radius: 14px; padding: 18px 20px; margin-bottom: 14px;}
.cc-card.High {border-left-color:#dc2626;} .cc-card.Medium {border-left-color:#d97706;} .cc-card.Low {border-left-color:#16a34a;}
.cc-head {display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;}
.cc-badge {padding:4px 12px; border-radius:999px; font-size:12px; font-weight:700;}
.cc-badge.High {background:#fee2e2;} .cc-badge.Medium {background:#fef3c7;} .cc-badge.Low {background:#dcfce7;}
.cc-quote {background:#f8fafc; border-radius:10px; padding:10px 14px; font-style:italic; margin-bottom:10px;}
.cc-ask {background:#eff6ff; border-radius:10px; padding:10px 14px; margin-top:10px;}
.cc-warn {background:#fff7ed; border:1px solid #fdba74; border-radius:10px; padding:8px 12px; margin-top:10px; font-size:13px;}
.cc-note {background:#fffbeb; border:1px solid #fde68a; border-radius:12px; padding:12px 14px; font-size:13px;}
@media (max-width: 700px) {.cc-stats {grid-template-columns: repeat(2,1fr);}}
"""

DARK = "color:#1f2937;"      # normal text
HEAD = "color:#111827;"      # titles
GREY = "color:#6b7280;"      # labels
HERO = """
<div class="cc-hero">
  <h1 style="margin:0;font-size:36px;font-weight:800;color:#ffffff;">🚨 ClauseCop</h1>
  <p style="margin:8px 0 16px;font-size:15px;color:#cbd5e1;">Understand your offer letter before you sign it. Upload the PDF and get a
     clause-by-clause risk review grounded in labour-law references.</p>
  <span class="cc-chip">Clause extraction</span><span class="cc-chip">Law-grounded RAG</span>
  <span class="cc-chip">High / Medium / Low risk</span><span class="cc-chip">English &amp; Tamil</span>
</div>
"""

DISCLAIMER = """<div class="cc-note" style="color:#92400e;"><b>Disclaimer:</b> ClauseCop provides awareness only and is not legal advice.
For important decisions, consult a qualified lawyer or the HR team.</div>"""

ICON = {"High": "🔴", "Medium": "🟡", "Low": "🟢"}
BADGE = {"High": "#b91c1c", "Medium": "#b45309", "Low": "#15803d"}
ORDER = {"High": 0, "Medium": 1, "Low": 2}
esc = html.escape


def notice(msg):
    return f'<div class="cc-note" style="color:#92400e;">{esc(msg)}</div>'


def build_summary(results):
    n = {k: sum(r["risk"] == k for r in results) for k in ORDER}
    if n["High"]:
        banner = ("#fee2e2", "#991b1b", f"{n['High']} high-risk clause(s) found. Review them carefully and clarify with HR before signing.")
    elif n["Medium"]:
        banner = ("#fef3c7", "#92400e", "Some clauses need clarification. Use the questions below when you speak to HR.")
    else:
        banner = ("#dcfce7", "#166534", "No major risks detected. Still read every clause carefully before signing.")
    box = lambda num, label, col: (f'<div class="cc-stat"><div style="font-size:30px;font-weight:800;color:{col};">{num}</div>'
                                   f'<div style="font-size:12px;letter-spacing:.5px;text-transform:uppercase;{GREY}">{label}</div></div>')
    return (f'<div class="cc-stats">{box(len(results), "Clauses", "#111827")}{box(n["High"], "High risk", "#dc2626")}'
            f'{box(n["Medium"], "Medium risk", "#d97706")}{box(n["Low"], "Low risk", "#16a34a")}</div>'
            f'<div class="cc-banner" style="background:{banner[0]};color:{banner[1]};">{banner[2]}</div>')


def label(text):
    return f'<div style="font-size:11px;font-weight:700;letter-spacing:.6px;text-transform:uppercase;margin:12px 0 3px;{GREY}">{text}</div>'


def build_cards(results, lang):
    out = []
    for r in sorted(results, key=lambda x: (ORDER[x["risk"]], x["id"])):
        cat = esc(r["category"].replace("_", " ").title())
        body = f'<div class="cc-quote" style="color:#475569;">{esc(r["text"])}</div>'
        body += label("Why this rating") + f'<div style="{DARK}">{esc(r["reason"])}</div>'
        if not r.get("verified", True):
            body += ('<div class="cc-warn" style="color:#9a3412;">⚠️ This explanation could not be verified against the '
                     'clause text. Please review this clause manually.</div>')
        if lang in ("English", "English + Tamil") and r["explanation_en"]:
            body += label("In simple terms") + f'<div style="{DARK}">{esc(r["explanation_en"])}</div>'
        if lang in ("Tamil", "English + Tamil") and r["explanation_ta"]:
            body += label("எளிய விளக்கம்") + f'<div style="{DARK}">{esc(r["explanation_ta"])}</div>'
        if r["ask_hr"]:
            body += f'<div class="cc-ask" style="color:#1e3a8a;">💬 <b>Ask HR:</b> {esc(r["ask_hr"])}</div>'
        refs = "".join(f'<p style="margin:6px 0;{GREY}"><b>{esc(x["source"])}</b>: {esc(x["text"][:280])}</p>' for x in r["references"])
        body += f'<details style="margin-top:10px;font-size:13px;{GREY}"><summary>Reference rules used</summary>{refs}</details>'
        out.append(
            f'<div class="cc-card {r["risk"]}"><div class="cc-head">'
            f'<span style="font-weight:700;font-size:16px;{HEAD}">Clause {r["id"]} · {cat}</span>'
            f'<span class="cc-badge {r["risk"]}" style="color:{BADGE[r["risk"]]};">{ICON[r["risk"]]} {r["risk"]} risk</span>'
            f'</div>{body}</div>'
        )

    qs = [r["ask_hr"] for r in sorted(results, key=lambda x: ORDER[x["risk"]])
          if r["risk"] in ("High", "Medium") and r["ask_hr"]]
    if qs:
        items = "".join(f'<li style="margin-bottom:6px;{DARK}">{esc(q)}</li>' for q in qs)
        out.append(f'<div class="cc-card" style="border-left-color:#1e3a8a;"><div style="font-weight:700;font-size:16px;{HEAD}">'
                   f'❓ Questions to ask HR before signing</div><ol style="margin-top:10px;">{items}</ol></div>')
    return "".join(out)


def analyze(pdf, lang, progress=gr.Progress()):
    if pdf is None:
        return "", notice("Please upload an offer letter PDF first.")
    try:
        progress(0.1, desc="Reading the document...")
        text = pdf_to_text(getattr(pdf, "name", pdf))
        progress(0.35, desc="Splitting into clauses...")
        clauses = extract_clauses(text)
        progress(0.65, desc="Checking clauses against labour-law references...")
        results = check_all(clauses)
        progress(1.0, desc="Done")
        return build_summary(results), build_cards(results, lang)
    except Exception as e:
        return "", notice(f"Analysis failed: {safe_err(e)}")


FORCE_LIGHT = "() => { document.body.classList.remove('dark'); }"

with gr.Blocks(css=CSS, theme=gr.themes.Soft(primary_hue="red", neutral_hue="slate"),
               title="ClauseCop", js=FORCE_LIGHT) as demo:
    gr.HTML(HERO)
    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            pdf_in = gr.File(label="Upload offer letter (PDF)", file_types=[".pdf"])
            lang_in = gr.Radio(["English", "Tamil", "English + Tamil"], value="English + Tamil", label="Explanation language")
            go = gr.Button("Analyze Document", variant="primary", size="lg")
            gr.HTML(DISCLAIMER)
        with gr.Column(scale=2):
            summary_out = gr.HTML()
            cards_out = gr.HTML()
    gr.HTML(f'<div style="text-align:center;font-size:12px;margin-top:18px;{GREY}">ClauseCop · AI-powered offer letter risk analysis · Awareness only, not legal advice</div>')
    go.click(analyze, [pdf_in, lang_in], [summary_out, cards_out])

_, local_url, share_url = demo.launch(share=True, prevent_thread_lock=True, quiet=True)
print("✅ ClauseCop is live. Open this link:", share_url)

/tmp/ipykernel_534/2593514172.py:126: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft(primary_hue="red", neutral_hue="slate"),
/tmp/ipykernel_534/2593514172.py:126: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft(primary_hue="red", neutral_hue="slate"),
/tmp/ipykernel_534/2593514172.py:126: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft(primary_hue="red", neutral_hue="slate"),


* Running on public URL: https://ba9c4b8431882731da.gradio.live


✅ ClauseCop is live. Open this link: https://ba9c4b8431882731da.gradio.live
